# IMFER: Interpretable Multimodal Fusion for Emotion Recognition
## End-to-End Reproducibility Pipeline — IEMOCAP · MELD · EmoryNLP

---

**Paper:** *IMFER: Interpretable Multimodal Fusion for Emotion Recognition in Conversations*

This notebook provides a fully reproducible, end-to-end experimental pipeline for all three benchmark datasets used in our study. Each stage applies an identical workflow across datasets, ensuring transparency and scientific rigor.

---

### Outline

| Stage | Description |
|:-----:|-------------|
| **1** | Environment Configuration & Shared Utilities |
| **2** | Dataset Integrity Verification (IEMOCAP, MELD, EmoryNLP) |
| **3** | Codebase Validation — Compilation & Unit Tests |
| **4** | Model Training — Sequential Execution Across All Datasets |
| **5** | Post-Training Evaluation — Metrics, Bootstrap CI, Visualizations |
| **6** | Consolidated Results & Artifact Summary |

---

### Evaluation Protocol Per Dataset

| Step | Output |
|------|--------|
| Train IMFER model | Best checkpoint per seed |
| Aggregate evaluation | Weighted-F1, Macro-F1, Accuracy |
| Bootstrap confidence intervals | 95% CI for all metrics |
| Visualization | Confusion matrix, per-class F1 bar plot |
| Classification report | Precision, recall, F1 per emotion class |

---

> **Instructions:** Execute all cells sequentially from top to bottom. The pipeline supports two profiles — *Quick* (single seed, bounded epochs) for demonstration, and *Full* (multi-seed) for complete reproduction.

---
## Stage 1: Environment Configuration & Shared Utilities

> **Objective:** Establish a unified configuration and helper functions used uniformly across IEMOCAP, MELD, and EmoryNLP.

This cell defines:
- Dataset metadata (label sets, class counts, file paths)
- Run profile selection (Quick vs. Full reproduction)
- Utility functions for command execution, training argument construction, and metadata generation from alignment files

In [ ]:
import json
import os
import pickle
import re
import csv
import subprocess
import sys
from pathlib import Path
from IPython.display import display, Image

# ═══════════════════════════════════════════════════════════════════════════════
# Configuration
# ═══════════════════════════════════════════════════════════════════════════════

ROOT = Path(os.getcwd())
PYTHON = r'C:\Users\uie72691\AppData\Local\Programs\Python\Python311\python.exe'

# ── Run Profile ──
# Set QUICK_PROFILE = True for a single-seed demonstration run
# Set QUICK_PROFILE = False for full multi-seed reproduction
QUICK_PROFILE = True

PROFILE = {
    'seeds': '42' if QUICK_PROFILE else None,
    'max_epochs': '20' if QUICK_PROFILE else None,
    'patience': '5' if QUICK_PROFILE else None,
}

DATASETS = {
    'iemocap': {
        'folder': 'IEMOCAP',
        'num_classes': 6,
        'class_names': 'happy,sad,neutral,angry,excited,frustrated',
        'metadata': ROOT / 'datasets' / 'IEMOCAP' / 'metadata.csv',
    },
    'meld': {
        'folder': 'MELD',
        'num_classes': 7,
        'class_names': 'neutral,surprise,fear,sadness,joy,disgust,anger',
        'metadata': ROOT / 'datasets' / 'MELD' / 'metadata.csv',
    },
    'emorynlp': {
        'folder': 'EmoryNLP',
        'num_classes': 7,
        'class_names': 'joyful,peaceful,powerful,scared,mad,sad,neutral',
        'metadata': ROOT / 'datasets' / 'EmoryNLP' / 'metadata.csv',
    },
}

# ═══════════════════════════════════════════════════════════════════════════════
# Shared Utilities
# ═══════════════════════════════════════════════════════════════════════════════

def stage(title):
    print('\n' + '=' * 80)
    print(f'  {title}')
    print('=' * 80)


def run_cmd(args, check=True):
    print(f'\n  $ {" ".join(map(str, args))}')
    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'
    result = subprocess.run(args, cwd=ROOT, text=True, capture_output=True, env=env)
    if result.stdout:
        for line in result.stdout.strip().split('\n'):
            print(f'    {line}')
    if result.stderr:
        for line in result.stderr.strip().split('\n'):
            print(f'    [stderr] {line}')
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed (exit {result.returncode}): {" ".join(map(str, args))}')
    return result


def _split_name(dataset, base_split):
    if base_split == 'valid':
        return 'dev' if dataset in {'meld', 'emorynlp'} else 'val'
    return base_split


def _infer_conversation_turn(utterance_id, fallback_index):
    utt = str(utterance_id)
    if '_' in utt:
        conv = utt.rsplit('_', 1)[0]
        tail = utt.rsplit('_', 1)[1]
    else:
        conv = utt
        tail = utt
    m = re.search(r'(\d+)$', tail)
    turn_idx = int(m.group(1)) if m else fallback_index
    return conv, turn_idx


def ensure_align_and_metadata(dataset):
    cfg = DATASETS[dataset]
    datasets_dir = ROOT / 'datasets' / cfg['folder']
    split_to_file = {'train': 'train_align.pkl', 'valid': 'valid_align.pkl', 'test': 'test_align.pkl'}
    all_rows = []

    for split, file_name in split_to_file.items():
        align_path = datasets_dir / file_name
        if not align_path.exists():
            raise FileNotFoundError(f'Missing align file for {dataset} split {split}: {align_path}')
        with open(align_path, 'rb') as f:
            items = pickle.load(f)
        out_split = _split_name(dataset, split)
        for idx, item in enumerate(items):
            if not isinstance(item, tuple) or len(item) < 3:
                continue
            payload, label, utterance_id = item[0], str(item[1]).lower(), str(item[2])
            text = ''
            if isinstance(payload, tuple) and len(payload) >= 4 and isinstance(payload[3], str):
                text = payload[3].strip()
            conv_id, turn_index = _infer_conversation_turn(utterance_id, idx)
            all_rows.append({
                'split': out_split, 'conversation_id': conv_id, 'turn_index': turn_index,
                'utterance_id': utterance_id, 'speaker_id': 'unknown', 'text': text,
                'audio_path': '', 'video_path': '', 'label': label,
            })

    metadata_path = cfg['metadata']
    metadata_path.parent.mkdir(parents=True, exist_ok=True)
    with open(metadata_path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=[
            'split', 'conversation_id', 'turn_index', 'utterance_id',
            'speaker_id', 'text', 'audio_path', 'video_path', 'label'
        ])
        writer.writeheader()
        writer.writerows(all_rows)
    print(f'    Wrote {metadata_path} with {len(all_rows)} rows')
    return metadata_path


def find_predictions_csv(dataset):
    pred_files = sorted((ROOT / 'artifacts' / dataset).glob('seed_*/predictions/test_predictions.csv'))
    return pred_files[0] if pred_files else None


def training_args(dataset):
    args = [PYTHON, 'train.py', '--dataset', dataset, '--device', 'cpu']
    if PROFILE['seeds']:
        args += ['--seeds', PROFILE['seeds']]
    if PROFILE['max_epochs']:
        args += ['--max_epochs', PROFILE['max_epochs']]
    if PROFILE['patience']:
        args += ['--patience', PROFILE['patience']]
    return args


# ═══════════════════════════════════════════════════════════════════════════════
stage('Stage 1: Environment configured')
print(f'  Root:    {ROOT}')
print(f'  Python:  {PYTHON}')
print(f'  Profile: {"Quick (single seed, bounded epochs)" if QUICK_PROFILE else "Full (multi-seed)"}')
print(f'  Datasets: {", ".join(DATASETS.keys())}')

---
## Stage 2: Dataset Integrity Verification

> **Objective:** Validate that all required alignment and metadata files are present and correctly structured for all three benchmark datasets.

Checks performed:
- Existence of `train_align.pkl`, `valid_align.pkl`, `test_align.pkl` per dataset
- Automatic generation of normalized `metadata.csv` from alignment data
- Readiness report with pass/fail status for each component

In [ ]:
stage('Stage 2: Dataset setup and readiness checks')

status = {}
for dataset, cfg in DATASETS.items():
    metadata_path = ensure_align_and_metadata(dataset)
    ds_dir = ROOT / 'datasets' / cfg['folder']
    status[f'{dataset}_metadata'] = metadata_path.exists()
    status[f'{dataset}_train_align'] = (ds_dir / 'train_align.pkl').exists()
    status[f'{dataset}_valid_align'] = (ds_dir / 'valid_align.pkl').exists()
    status[f'{dataset}_test_align'] = (ds_dir / 'test_align.pkl').exists()

print('\nReadiness status:')
print(json.dumps(status, indent=2))

missing = [k for k, v in status.items() if not v]
if missing:
    raise FileNotFoundError('Missing required dataset files: ' + ', '.join(missing))

---
## Stage 3: Codebase Validation — Compilation & Unit Tests

> **Objective:** Ensure code integrity before training by running syntax checks and the test suite.

Validations:
- Python `compileall` — verifies all modules parse without errors
- Unit tests (`tests/test_data_pipeline.py`) — confirms data loading, split mapping, and label consistency

In [ ]:
stage('Stage 3: Build and pipeline validation')
py_files = [str(f) for f in ROOT.glob('*.py')]
run_cmd([PYTHON, '-m', 'compileall'] + py_files)
run_cmd([PYTHON, '-m', 'unittest', 'tests/test_data_pipeline.py', '-v'])

---
## Stage 4: Model Training — Sequential Execution

> **Objective:** Train the IMFER model on IEMOCAP, MELD, and EmoryNLP using identical hyperparameter profiles.

Training configuration:
- **Quick profile:** seed=42, max 20 epochs, patience=5 (for demonstration)
- **Full profile:** 5 seeds (42, 123, 256, 512, 1024), default epochs and patience

Each training run produces checkpoints, predictions, and per-seed metrics under `artifacts/<dataset>/`.

In [ ]:
stage('Stage 4: Train all datasets one by one')

for dataset, cfg in DATASETS.items():
    if not cfg['metadata'].exists():
        raise FileNotFoundError(f'Missing metadata for {dataset}: {cfg["metadata"]}')

    # Skip training if predictions already exist from a prior run
    pred_csv = find_predictions_csv(dataset)
    if pred_csv is not None:
        print(f'\n[SKIP] {dataset}: Predictions already exist at {pred_csv.relative_to(ROOT)}')
        print(f'       To retrain, delete artifacts/{dataset}/seed_*/predictions/ and rerun.')
        continue

    print(f'\nTraining dataset: {dataset}')
    run_cmd(training_args(dataset))

---
## Stage 5: Post-Training Evaluation & Visualization

> **Objective:** Apply a uniform evaluation protocol to all datasets — producing metrics, statistical analysis, and publication-ready figures.

Operations per dataset:
1. **Aggregate evaluation** — Weighted-F1, Macro-F1, Accuracy across seeds
2. **Bootstrap analysis** — 95% confidence intervals via 1000-sample bootstrap
3. **Visualization** — Metric trends and seed-level comparisons
4. **Classification report** — Per-class precision, recall, F1-score
5. **Diagnostic plots** — Confusion matrix and per-class F1 bar charts

In [ ]:
stage('Stage 5: Identical post-training operations for all datasets')

for dataset, cfg in DATASETS.items():
    print(f'\nProcessing dataset: {dataset}')
    aggregate_dir = ROOT / 'artifacts' / dataset / 'aggregate'
    figures_dir = ROOT / 'figures' / dataset

    run_cmd([
        PYTHON, 'evaluate.py',
        '--artifacts_root', './artifacts',
        '--dataset', dataset,
        '--num_classes', str(cfg['num_classes'])
    ])

    run_cmd([
        PYTHON, 'bootstrap_analysis.py',
        '--aggregate_csv', f'./artifacts/{dataset}/aggregate/metrics.csv',
        '--out_json', f'./artifacts/{dataset}/aggregate/bootstrap_summary.json'
    ])

    run_cmd([
        PYTHON, 'visualize_results.py',
        '--aggregate_csv', f'./artifacts/{dataset}/aggregate/metrics.csv',
        '--output_dir', f'./figures/{dataset}'
    ])

    pred_csv = find_predictions_csv(dataset)
    if pred_csv is None:
        raise FileNotFoundError(f'No predictions CSV found for {dataset} under artifacts/{dataset}/seed_*/predictions/')

    run_cmd([
        PYTHON, 'classification_report_and_plots.py',
        '--predictions_csv', str(pred_csv.relative_to(ROOT)).replace('\\', '/'),
        '--class_names', cfg['class_names'],
        '--dataset_name', dataset,
        '--output_dir', f'./figures/{dataset}',
        '--report_out', f'./artifacts/{dataset}/aggregate/classification_report.txt',
        '--json_out', f'./artifacts/{dataset}/aggregate/classification_report.json'
    ])

    report_txt = aggregate_dir / 'classification_report.txt'
    if report_txt.exists():
        print(f'\n== {dataset.upper()} classification_report.txt ==')
        print(report_txt.read_text(encoding='utf-8'))

    for fig_name in ['confusion_matrix.png', 'per_class_f1.png']:
        fig_path = figures_dir / fig_name
        if fig_path.exists():
            print(f'Displaying: {fig_path.relative_to(ROOT)}')
            display(Image(filename=str(fig_path)))
        else:
            print(f'Missing expected figure: {fig_path}')

---
## Stage 6: Consolidated Results Summary

> **Objective:** Present all key metrics in a comparative table format — rows as evaluation metrics, columns as datasets — for streamlined review by panel members.

Tables generated:
1. **Primary Metrics Comparison** — WF1, MF1, Accuracy, Bootstrap CI
2. **Modality Contribution Scores** — Text, Audio, Visual contribution per dataset
3. **Per-Class F1 Scores** — Emotion-level performance per dataset

In [ ]:
stage('Stage 6: Consolidated results summary for all datasets')

from IPython.display import display, HTML

# ─── Load all summary data ───
summaries = {}
bootstraps = {}
class_reports = {}

for dataset in DATASETS.keys():
    summary_path = ROOT / 'artifacts' / dataset / 'aggregate' / 'summary.json'
    bootstrap_path = ROOT / 'artifacts' / dataset / 'aggregate' / 'bootstrap_summary.json'
    report_path = ROOT / 'artifacts' / dataset / 'aggregate' / 'classification_report.json'

    if summary_path.exists():
        summaries[dataset] = json.loads(summary_path.read_text(encoding='utf-8'))
    if bootstrap_path.exists():
        bootstraps[dataset] = json.loads(bootstrap_path.read_text(encoding='utf-8'))
    if report_path.exists():
        class_reports[dataset] = json.loads(report_path.read_text(encoding='utf-8'))

# ─── Table 1: Primary Metrics Comparison ───
print('\n')
datasets_list = list(DATASETS.keys())

table1_html = '''
<h3>Table 1: Primary Metrics Comparison</h3>
<table border=\"1\" cellpadding=\"8\" cellspacing=\"0\" style=\"border-collapse: collapse; text-align: center; font-size: 14px;\">
<tr style=\"background-color: #2c3e50; color: white;\">
    <th style=\"padding: 10px;\">Metric</th>'''
for ds in datasets_list:
    table1_html += f'<th style=\"padding: 10px;\">{ds.upper()}</th>'
table1_html += '</tr>'

metrics_rows = [
    ('Weighted F1 (%)', 'wf1_mean'),
    ('Macro F1 (%)', 'mf1'),
    ('Accuracy (%)', 'accuracy'),
    ('WF1 Std Dev', 'wf1_std'),
    ('Num Seeds', 'num_runs'),
]

for i, (label, key) in enumerate(metrics_rows):
    bg = '#f8f9fa' if i % 2 == 0 else '#ffffff'
    table1_html += f'<tr style=\"background-color: {bg};\"><td style=\"padding: 8px; font-weight: bold; text-align: left;\">{label}</td>'
    for ds in datasets_list:
        s = summaries.get(ds, {})
        if key == 'wf1_mean':
            val = f"{s.get('wf1_mean', 0):.2f}"
        elif key == 'wf1_std':
            val = f"{s.get('wf1_std', 0):.4f}"
        elif key == 'num_runs':
            val = str(s.get('num_runs', '-'))
        elif key in ('mf1', 'accuracy'):
            runs = s.get('runs', [])
            if runs:
                vals = [r.get(key, 0) for r in runs if r.get(key, 0) > 0]
                val = f"{sum(vals)/len(vals):.2f}" if vals else f"{runs[0].get(key, 0):.2f}"
            else:
                val = '-'
        else:
            val = '-'
        table1_html += f'<td style=\"padding: 8px;\">{val}</td>'
    table1_html += '</tr>'

# Add Bootstrap CI row
table1_html += f'<tr style=\"background-color: #f8f9fa;\"><td style=\"padding: 8px; font-weight: bold; text-align: left;\">Bootstrap 95% CI</td>'
for ds in datasets_list:
    b = bootstraps.get(ds, {})
    ci = b.get('bootstrap_ci95', [0, 0])
    table1_html += f'<td style=\"padding: 8px;\">[{ci[0]:.2f}, {ci[1]:.2f}]</td>'
table1_html += '</tr>'
table1_html += '</table>'

display(HTML(table1_html))

# ─── Table 2: Modality Contribution Scores (MCS) ───
table2_html = '''
<br>
<h3>Table 2: Modality Contribution Scores (MCS)</h3>
<table border=\"1\" cellpadding=\"8\" cellspacing=\"0\" style=\"border-collapse: collapse; text-align: center; font-size: 14px;\">
<tr style=\"background-color: #2c3e50; color: white;\">
    <th style=\"padding: 10px;\">Modality</th>'''
for ds in datasets_list:
    table2_html += f'<th style=\"padding: 10px;\">{ds.upper()}</th>'
table2_html += '</tr>'

modalities = [('Text', 'mcs_text'), ('Audio', 'mcs_audio'), ('Visual', 'mcs_visual')]
for i, (mod_name, mod_key) in enumerate(modalities):
    bg = '#f8f9fa' if i % 2 == 0 else '#ffffff'
    table2_html += f'<tr style=\"background-color: {bg};\"><td style=\"padding: 8px; font-weight: bold; text-align: left;\">{mod_name}</td>'
    for ds in datasets_list:
        s = summaries.get(ds, {})
        runs = s.get('runs', [])
        if runs and mod_key in runs[0]:
            val = f"{runs[0][mod_key]:.4f}"
        else:
            val = '-'
        table2_html += f'<td style=\"padding: 8px;\">{val}</td>'
    table2_html += '</tr>'
table2_html += '</table>'

display(HTML(table2_html))

# ─── Table 3: Per-Class F1 Scores ───
table3_html = '''
<br>
<h3>Table 3: Per-Class F1 Scores by Dataset</h3>
<table border=\"1\" cellpadding=\"8\" cellspacing=\"0\" style=\"border-collapse: collapse; text-align: center; font-size: 14px;\">
<tr style=\"background-color: #2c3e50; color: white;\">
    <th style=\"padding: 10px;\">Dataset</th>
    <th style=\"padding: 10px;\">Class</th>
    <th style=\"padding: 10px;\">Precision</th>
    <th style=\"padding: 10px;\">Recall</th>
    <th style=\"padding: 10px;\">F1-Score</th>
    <th style=\"padding: 10px;\">Support</th>
</tr>'''

for ds in datasets_list:
    report = class_reports.get(ds, {})
    class_names = report.get('class_names', [])
    class_metrics = report.get('class_metrics', [])
    for i, (cn, cm) in enumerate(zip(class_names, class_metrics)):
        bg = '#f8f9fa' if i % 2 == 0 else '#ffffff'
        ds_cell = f'<td rowspan=\"{len(class_names)}\" style=\"padding: 8px; font-weight: bold; vertical-align: middle;\">{ds.upper()}</td>' if i == 0 else ''
        table3_html += f'<tr style=\"background-color: {bg};\">{ds_cell}'
        table3_html += f'<td style=\"padding: 8px;\">{cn}</td>'
        table3_html += f'<td style=\"padding: 8px;\">{cm["precision"]:.4f}</td>'
        table3_html += f'<td style=\"padding: 8px;\">{cm["recall"]:.4f}</td>'
        table3_html += f'<td style=\"padding: 8px;\">{cm["f1"]:.4f}</td>'
        table3_html += f'<td style=\"padding: 8px;\">{cm["support"]}</td>'
        table3_html += '</tr>'
    # Add summary row for this dataset
    summary_data = report.get('summary', {})
    weighted = summary_data.get('weighted', {})
    overall = summary_data.get('overall', {})
    table3_html += f'<tr style=\"background-color: #eaf2f8; font-weight: bold;\">'
    table3_html += f'<td style=\"padding: 8px;\" colspan=\"2\">Weighted Avg / Accuracy</td>'
    table3_html += f'<td style=\"padding: 8px;\">{weighted.get("precision", 0):.4f}</td>'
    table3_html += f'<td style=\"padding: 8px;\">{weighted.get("recall", 0):.4f}</td>'
    table3_html += f'<td style=\"padding: 8px;\">{weighted.get("f1", 0):.4f}</td>'
    table3_html += f'<td style=\"padding: 8px;\">{overall.get("support", 0)}</td>'
    table3_html += '</tr>'

table3_html += '</table>'

display(HTML(table3_html))

# ─── Final Summary Statement ───
print('\n' + '=' * 80)
print('REPRODUCTION COMPLETE')
print('=' * 80)
print(f'\nDatasets evaluated: {", ".join(d.upper() for d in datasets_list)}')
print(f'Run profile: {"Quick (single seed)" if QUICK_PROFILE else "Full (multi-seed)"}')
print(f'\nArtifacts saved to: ./artifacts/<dataset>/aggregate/')
print(f'Figures saved to:   ./figures/<dataset>/')